# Fine-tuning Qwen3.5-0.8B with LoRA/QLoRA

This notebook demonstrates how to fine-tune the Qwen3.5-0.8B model on a downstream task using parameter-efficient methods (LoRA/QLoRA).

## Overview

- **Model**: Qwen3.5-0.8B (8-bit quantized)
- **Method**: QLoRA (Quantized Low-Rank Adaptation)
- **Memory**: ~6GB GPU RAM required
- **Use Cases**: Classification, QA, Text Generation, Instruction Following

## 1. Setup

In [ ]:
import torch
import json
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

from model.qwen_finetuner import (
    QwenFineTuner,
    FineTuningConfig,
    LoRAConfig,
    create_finetuning_config,
)
from data.finetune_datamodule import FinetuneDataModule, FinetuneDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 2. Prepare Training Data

Create a sample dataset for demonstration. In practice, you would load your own data.

In [ ]:
# Create sample instruction-tuning dataset
sample_data = [
    {
        "instruction": "Translate to French",
        "input": "Hello, how are you?",
        "output": "Bonjour, comment allez-vous?"
    },
    {
        "instruction": "Translate to French",
        "input": "I am fine, thank you.",
        "output": "Je vais bien, merci."
    },
    {
        "instruction": "Summarize the text",
        "input": "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet.",
        "output": "A sentence containing all alphabet letters describes a fox jumping over a dog."
    },
    {
        "instruction": "Answer the question",
        "input": "Context: Paris is the capital of France. Question: What is the capital of France?",
        "output": "The capital of France is Paris."
    },
    {
        "instruction": "Classify the sentiment",
        "input": "I love this product! It works amazingly well.",
        "output": "Positive"
    },
    {
        "instruction": "Classify the sentiment",
        "input": "This is the worst experience I've ever had.",
        "output": "Negative"
    },
    {
        "instruction": "Generate a greeting",
        "input": "",
        "output": "Hello! How can I assist you today?"
    },
    {
        "instruction": "What is machine learning?",
        "input": "",
        "output": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed."
    },
]

# Save to file
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "sample_instructions.json", "w") as f:
    json.dump(sample_data, f, indent=2)

print(f"Created sample dataset with {len(sample_data)} examples")
print(json.dumps(sample_data[0], indent=2))

## 3. Configure Fine-tuning

In [ ]:
# Create fine-tuning configuration
config = create_finetuning_config(
    model_name="unsloth/Qwen3.5-0.8B-Q8_0",  # Qwen3.5 8-bit quantized
    lora_rank=8,          # LoRA rank (higher = more parameters)
    lora_alpha=16,        # LoRA scaling factor
    learning_rate=2e-4,   # Learning rate
    num_epochs=3,         # Number of epochs
    batch_size=2,         # Batch size (reduce if OOM)
    use_quantization=True, # Use QLoRA (4-bit)
    use_lora=True,        # Use LoRA adapters
    max_length=512,       # Max sequence length
    gradient_accumulation_steps=4,  # Effective batch = 2 * 4 = 8
)

print("Fine-tuning Configuration:")
print(f"  Model: {config.model_name}")
print(f"  LoRA: r={config.lora.r}, alpha={config.lora.lora_alpha}")
print(f"  Quantization: {config.use_quantization}")
print(f"  Learning rate: {config.learning_rate}")
print(f"  Batch size: {config.batch_size}")
print(f"  Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
print(f"  Epochs: {config.num_epochs}")
print(f"  Max length: {config.max_length}")

## 4. Load Model and Tokenizer

In [ ]:
# Initialize the fine-tuner
model = QwenFineTuner(config=config)

# Load and configure the model
print("Loading model and applying LoRA...")
model.setup_model()

print("\nModel loaded successfully!")

## 5. Setup Data Module

In [ ]:
# Create data module
datamodule = FinetuneDataModule(
    data_path=str(data_dir / "sample_instructions.json"),
    model_name=config.model_name,
    max_length=config.max_length,
    batch_size=config.batch_size,
    num_workers=0,
    seed=42,
    format_type="instruction",
    validation_split=0.2,
)

# Setup (load data and tokenizer)
datamodule.setup()

# Print dataset info
info = datamodule.get_dataset_info()
print("Dataset Information:")
print(f"  Training samples: {info['train_samples']}")
print(f"  Validation samples: {info['val_samples']}")
print(f"  Vocabulary size: {info['vocab_size']:,}")

## 6. Training

Note: For this demo, we'll do a quick training run. In practice, you would train for more epochs.

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from datetime import datetime

# Create output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path("outputs") / f"finetuned_notebook_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

# Setup callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir / "checkpoints",
    filename="checkpoint-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_last=True,
    save_top_k=1,
)

lr_monitor = LearningRateMonitor(logging_interval="step")

# Create trainer
trainer = L.Trainer(
    max_epochs=config.num_epochs,
    accelerator="auto",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    gradient_clip_val=1.0,
    accumulate_grad_batches=config.gradient_accumulation_steps,
    callbacks=[checkpoint_callback, lr_monitor],
    default_root_dir=output_dir,
    val_check_interval=1.0,  # Validate every epoch
    enable_progress_bar=True,
)

print(f"Output directory: {output_dir}")
print(f"Starting training for {config.num_epochs} epochs...")

In [ ]:
# Train the model
trainer.fit(model, datamodule=datamodule)

print("\nTraining complete!")

## 7. Save the Fine-tuned Model

In [ ]:
# Save the fine-tuned model
model.save_model(output_dir / "adapter_model")

# Save training configuration
training_config = {
    "model_name": config.model_name,
    "use_lora": config.use_lora,
    "use_quantization": config.use_quantization,
    "lora_config": {
        "r": config.lora.r,
        "lora_alpha": config.lora.lora_alpha,
        "lora_dropout": config.lora.lora_dropout,
    },
    "training_config": {
        "learning_rate": config.learning_rate,
        "num_epochs": config.num_epochs,
        "batch_size": config.batch_size,
        "max_length": config.max_length,
    },
}

with open(output_dir / "finetune_config.json", "w") as f:
    json.dump(training_config, f, indent=2)

print(f"Model saved to: {output_dir}")
print(f"\nContents:")
for item in output_dir.iterdir():
    print(f"  - {item.name}")

## 8. Inference

In [ ]:
# Test generation with the fine-tuned model
test_prompts = [
    "Instruction: Translate to Spanish\nInput: Good morning!\nResponse:",
    "Instruction: What is AI?\nInput: \nResponse:",
]

print("Testing inference with fine-tuned model:\n")
print("=" * 60)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 40)
    
    try:
        response = model.generate(
            input_text=prompt,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
        )
        print(f"Response: {response}")
    except Exception as e:
        print(f"Error: {e}")

print("\n" + "=" * 60)

## Summary

This notebook demonstrated:

1. **Setup**: Loading the Qwen3.5-0.8B model with QLoRA
2. **Data Preparation**: Creating instruction-tuning datasets
3. **Configuration**: Setting up LoRA parameters
4. **Training**: Fine-tuning with PyTorch Lightning
5. **Inference**: Generating text with the fine-tuned model

### Next Steps

- Increase training data for better results
- Tune hyperparameters (learning rate, LoRA rank, epochs)
- Use the CLI script for full training: `python src/train_qwen_finetune.py`
- Run inference with: `python src/run_qwen_finetuned_inference.py`